In [1]:
import os
import pandas as pd
from pypdf import PdfReader

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:
pdf_path = "../dataset/sample.pdf"

reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    text += page.extract_text() + "\n"

with open("../dataset/extracted_text.txt", "w", encoding="utf-8") as file:
    file.write(text)

print("PDF text extracted successfully!")
print("Total pages:", len(reader.pages))
print(text[:1000])

PDF text extracted successfully!
Total pages: 20
Artificial Intelligence Study Guide
Page 1 of 20
 ARTIFICIAL INTELLIGENCE
A 20-Page Introductory Study Guide
 Concepts  Methods  Applications  Responsible Use
Purpose
A concise learning resource for exploring artificial intelligence, machine learning, language models,
semantic search, and retrieval-augmented generation.
 Prepared as a sample document for PDF question-answering practice.

Artificial Intelligence Study Guide
Page 2 of 20
A Brief History of Artificial Intelligence
The idea of artificial beings and mechanical reasoning appears in stories and philosophy long before
modern computers. AI became a formal research area during the middle of the twentieth century, when
researchers began asking whether machines could carry out reasoning and problem-solving tasks.
In 1950, Alan Turing proposed a practical discussion of machine intelligence that later became
associated with the Turing Test. In 1956, the Dartmouth research workshop 

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_text(text)

print("Total chunks:", len(chunks))
print("First chunk:")
print(chunks[0])

Total chunks: 58
First chunk:
Artificial Intelligence Study Guide
Page 1 of 20
 ARTIFICIAL INTELLIGENCE
A 20-Page Introductory Study Guide
 Concepts  Methods  Applications  Responsible Use
Purpose
A concise learning resource for exploring artificial intelligence, machine learning, language models,
semantic search, and retrieval-augmented generation.
 Prepared as a sample document for PDF question-answering practice.


In [4]:
chunks_df = pd.DataFrame({
    "chunk_id": range(1, len(chunks) + 1),
    "text": chunks
})

chunks_df.to_csv("../dataset/chunks.csv", index=False)

print("Chunks saved successfully!")
print(chunks_df.head())

Chunks saved successfully!
   chunk_id                                               text
0         1  Artificial Intelligence Study Guide\nPage 1 of...
1         2  Artificial Intelligence Study Guide\nPage 2 of...
2         3  In 1950, Alan Turing proposed a practical disc...
3         4  data, and methods were often insufficient for ...
4         5  Artificial Intelligence Study Guide\nPage 3 of...


In [6]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

vectorstore = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    persist_directory="../chroma_db",
    collection_name="pdf_collection"
)

print("ChromaDB created successfully!")

ChromaDB created successfully!


In [7]:
queries = [
    "What is artificial intelligence?",
    "What is machine learning?",
    "What is deep learning?",
    "What are neural networks?",
    "What is natural language processing?",
    "What is supervised learning?",
    "What is unsupervised learning?",
    "What is reinforcement learning?",
    "What are the applications of AI?",
    "What is data science?",
    "What are training datasets?",
    "What is a machine learning model?",
    "What is computer vision?",
    "What are the benefits of AI?",
    "What are the challenges of AI?"
]

print("Total questions:", len(queries))

Total questions: 15


In [8]:
queries_df = pd.DataFrame({
    "query_id": range(1, len(queries) + 1),
    "question": queries
})

queries_df.to_csv(
    "../dataset/queries.csv",
    index=False
)

print("Questions saved successfully!")
print(queries_df)

Questions saved successfully!
    query_id                              question
0          1      What is artificial intelligence?
1          2             What is machine learning?
2          3                What is deep learning?
3          4             What are neural networks?
4          5  What is natural language processing?
5          6          What is supervised learning?
6          7        What is unsupervised learning?
7          8       What is reinforcement learning?
8          9      What are the applications of AI?
9         10                 What is data science?
10        11           What are training datasets?
11        12     What is a machine learning model?
12        13              What is computer vision?
13        14          What are the benefits of AI?
14        15        What are the challenges of AI?


In [9]:
all_results = []

for query_id, query in enumerate(queries, 1):

    results = vectorstore.similarity_search_with_score(
        query,
        k=5
    )

    for rank, (doc, score) in enumerate(results, 1):

        all_results.append({
            "query_id": query_id,
            "question": query,
            "rank": rank,
            "retrieved_chunk": doc.page_content,
            "similarity_score": score
        })

print("Top-5 retrieval completed!")
print("Total retrieved chunks:", len(all_results))

Top-5 retrieval completed!
Total retrieved chunks: 75


In [11]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2", temperature=0)

answers = []

for query_id, query in enumerate(queries, 1):

    results = vectorstore.similarity_search(query, k=5)

    context = "\n".join(
        [doc.page_content for doc in results]
    )

    prompt = f"""
    Answer the question using only the following PDF content.
    If the answer is not available, say so.

    PDF Content:
    {context}

    Question: {query}
    """

    response = llm.invoke(prompt)

    answers.append({
        "query_id": query_id,
        "question": query,
        "answer": response.content
    })

print("RAG answers generated successfully!")

RAG answers generated successfully!


In [12]:
answers_df = pd.DataFrame(answers)

answers_df.to_csv(
    "../dataset/answers.csv",
    index=False
)

print("Answers saved successfully!")
print(answers_df.head())

Answers saved successfully!
   query_id                              question  \
0         1      What is artificial intelligence?   
1         2             What is machine learning?   
2         3                What is deep learning?   
3         4             What are neural networks?   
4         5  What is natural language processing?   

                                              answer  
0  According to the PDF content, Artificial Intel...  
1  According to the PDF content, Machine Learning...  
2  According to the PDF content, deep learning is...  
3  According to the PDF content, neural networks ...  
4  According to the PDF content, Natural Language...  


In [13]:
normal_answers = []

for query_id, query in enumerate(queries, 1):

    prompt = f"Answer this question clearly: {query}"

    response = llm.invoke(prompt)

    normal_answers.append({
        "query_id": query_id,
        "question": query,
        "normal_answer": response.content
    })

print("Normal answers generated successfully!")

Normal answers generated successfully!


In [ ]:
comparison_df = pd.DataFrame({
    "query_id": range(1, len(queries) + 1),
    "question": queries,
    "normal_answer": [a["normal_answer"] for a in normal_answers],
    "rag_answer": [a["answer"] for a in answers]
})

comparison_df.to_csv(
    "../dataset/answer_comparison.csv",
    index=False
)

print("Answer comparison saved successfully!")
print(comparison_df.head())

In [14]:
comparison_df = pd.DataFrame({
    "query_id": range(1, len(queries) + 1),
    "question": queries,
    "normal_answer": [a["normal_answer"] for a in normal_answers],
    "rag_answer": [a["answer"] for a in answers]
})

comparison_df.to_csv(
    "../dataset/answer_comparison.csv",
    index=False
)

print("Answer comparison saved successfully!")
print(comparison_df.head())

Answer comparison saved successfully!
   query_id                              question  \
0         1      What is artificial intelligence?   
1         2             What is machine learning?   
2         3                What is deep learning?   
3         4             What are neural networks?   
4         5  What is natural language processing?   

                                       normal_answer  \
0  Artificial intelligence (AI) refers to the dev...   
1  Machine learning (ML) is a subset of artificia...   
2  Deep learning is a subset of machine learning ...   
3  A neural network is a computer system inspired...   
4  Natural Language Processing (NLP) is a subfiel...   

                                          rag_answer  
0  According to the PDF content, Artificial Intel...  
1  According to the PDF content, Machine Learning...  
2  According to the PDF content, deep learning is...  
3  According to the PDF content, neural networks ...  
4  According to the PDF content

In [15]:
chunk_sizes = [300, 500, 1000]
overlap = 100

for size in chunk_sizes:

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=overlap
    )

    test_chunks = splitter.split_text(text)

    print("Chunk size:", size)
    print("Total chunks:", len(test_chunks))
    print("----------------------")

Chunk size: 300
Total chunks: 108
----------------------
Chunk size: 500
Total chunks: 58
----------------------
Chunk size: 1000
Total chunks: 38
----------------------


In [16]:
experiment_results = []

for size in chunk_sizes:

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=overlap
    )

    test_chunks = splitter.split_text(text)

    experiment_results.append({
        "chunk_size": size,
        "chunk_overlap": overlap,
        "total_chunks": len(test_chunks)
    })

experiment_df = pd.DataFrame(experiment_results)

experiment_df.to_csv(
    "../dataset/chunk_experiments.csv",
    index=False
)

print("Chunk experiments saved successfully!")
print(experiment_df)

Chunk experiments saved successfully!
   chunk_size  chunk_overlap  total_chunks
0         300            100           108
1         500            100            58
2        1000            100            38


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

keyword_results = []

for query_id, query in enumerate(queries, 1):

    vectorizer = TfidfVectorizer()

    tfidf_matrix = vectorizer.fit_transform(chunks)

    query_vector = vectorizer.transform([query])

    scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    top_indices = scores.argsort()[-5:][::-1]

    for rank, index in enumerate(top_indices, 1):

        keyword_results.append({
            "query_id": query_id,
            "question": query,
            "rank": rank,
            "retrieved_chunk": chunks[index],
            "similarity_score": scores[index]
        })

print("Keyword search completed!")
print("Total results:", len(keyword_results))

Keyword search completed!
Total results: 75


In [18]:
keyword_df = pd.DataFrame(keyword_results)

keyword_df.to_csv(
    "../dataset/keyword_search_results.csv",
    index=False
)

comparison = []

for query_id, query in enumerate(queries, 1):

    semantic = [
        r["retrieved_chunk"]
        for r in all_results
        if r["query_id"] == query_id
    ]

    keyword = [
        r["retrieved_chunk"]
        for r in keyword_results
        if r["query_id"] == query_id
    ]

    common = set(semantic) & set(keyword)

    comparison.append({
        "query_id": query_id,
        "question": query,
        "semantic_results": " | ".join(semantic),
        "keyword_results": " | ".join(keyword),
        "common_chunks": len(common)
    })

comparison_df = pd.DataFrame(comparison)

comparison_df.to_csv(
    "../dataset/semantic_vs_keyword_comparison.csv",
    index=False
)

print("Comparison completed successfully!")
print(comparison_df.head())

Comparison completed successfully!
   query_id                              question  \
0         1      What is artificial intelligence?   
1         2             What is machine learning?   
2         3                What is deep learning?   
3         4             What are neural networks?   
4         5  What is natural language processing?   

                                    semantic_results  \
0  Artificial Intelligence Study Guide\nPage 3 of...   
1  Artificial Intelligence Study Guide\nPage 5 of...   
2  Deep learning is a subfield of machine learnin...   
3  Artificial Intelligence Study Guide\nPage 9 of...   
4  Artificial Intelligence Study Guide\nPage 10 o...   

                                     keyword_results  common_chunks  
0  Artificial Intelligence Study Guide\nPage 5 of...              3  
1  Deep learning is a subfield of machine learnin...              4  
2  Deep learning is a subfield of machine learnin...              2  
3  how changes in weights aff

In [19]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")

result = embeddings.embed_query("What is artificial intelligence?")

print("Embedding successful!")
print("Vector length:", len(result))

Embedding successful!
Vector length: 768


In [20]:
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

vectorstore = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    persist_directory="../chroma_db",
    collection_name="pdf_collection"
)

print("ChromaDB created successfully!")

ChromaDB created successfully!


In [21]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")

result = embeddings.embed_query("Hello AI")

print("Embedding working!")
print("Vector length:", len(result))

Embedding working!
Vector length: 768


In [ ]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

result = embeddings.embed_query(
    "What is artificial intelligence?"
)

print("Embedding successful!")
print("Vector length:", len(result))

In [10]:
results_df = pd.DataFrame(all_results)

results_df.to_csv(
    "../dataset/search_results.csv",
    index=False
)

print("Search results saved successfully!")
print(results_df.head())

Search results saved successfully!
   query_id                          question  rank  \
0         1  What is artificial intelligence?     1   
1         1  What is artificial intelligence?     2   
2         1  What is artificial intelligence?     3   
3         1  What is artificial intelligence?     4   
4         1  What is artificial intelligence?     5   

                                     retrieved_chunk  similarity_score  
0  Artificial Intelligence Study Guide\nPage 3 of...          0.403754  
1  Artificial Intelligence Study Guide\nPage 5 of...          0.425638  
2  Artificial Intelligence Study Guide\nPage 20 o...          0.428300  
3  Artificial Intelligence Study Guide\nPage 2 of...          0.507937  
4  Artificial Intelligence Study Guide\nPage 4 of...          0.521440  
